# Main Prototype

## Install dependencies

In [ ]:
!git clone https://github.com/Keeron1/skeleton-based-hoi-recognition.git
%cd /content/skeleton-based-hoi-recognition

Cloning into 'skeleton-based-hoi-recognition'...


In [ ]:
%pip install -r requirements.txt

# Run Imports

In [ ]:
import sys
from pathlib import Path
import os
import cv2
import glob

In [ ]:
# Set execution root in the project root
sys.path.insert(0, str(Path.cwd().parent))

In [ ]:
from src.utils.config import Config
from src.utils.draw_boxes import DrawBoxes
from src.detector.yolo_detector import YOLODetector
from src.tracker.deepsort_tracker import DeepSortTracker

# Config


## Setup Env variables for Google Colab

In [ ]:
# Set Colab env vars
os.environ["ENV"] = "colab"
os.environ["DATA_ROOT"] = "/content/drive/MyDrive/HOI-Dataset"
from google.colab.patches import cv2_imshow # cv2.imshow("title", frame) doesn't work in Colab so this is the fix cv2_imshow(frame)

## Initalize config

In [ ]:
cfg = Config() # Path.cwd().parent

PROJECT_ROOT = cfg.project_root

dataset_path = cfg.get("paths", "dataset.raw")

output_path = PROJECT_ROOT / cfg.get("paths", "outputs.videos")
models_dir = PROJECT_ROOT / cfg.get("paths", "detector")
yolo_model_type = models_dir / cfg.get("model", "yolo.model_type") # Set path where model will load or download from

# Dataset

# YOLO

In [ ]:
# Load model
detector = YOLODetector(
    model_type_path=yolo_model_type,
    project_root=PROJECT_ROOT,
)

## Train

In [ ]:
results = YOLODetector.train(dataset_path)

## Test

In [ ]:
frame = cv2.imread("test.jpg")
results = detector.predict(frame)

print(results)

In [ ]:
# Process results list
for result in results[:10]:
    boxes = result.boxes  # Boxes object for bounding box outputs
    # print(boxes.data.tolist()) # x1, y1, x2, y2, conf score, class id
    masks = result.masks  # Masks object for segmentation masks outputs
    keypoints = result.keypoints  # Keypoints object for pose outputs
    probs = result.probs  # Probs object for classification outputs
    obb = result.obb  # Oriented boxes object for OBB outputs
    result.show()  # display to screen
    # result.save(filename="result.jpg")  # save to disk

# DeepSORT

In [ ]:
tracker = DeepSortTracker()

In [ ]:
detections = tracker.yolo_to_deepsort(results)

tracks = tracker.update(detections, frame)

for track in tracks:
    if not track.is_confirmed():
        continue

    track_id = track.track_id
    bbox = track.to_ltrb()

    print(track_id, bbox)

## Test DeepSORT

In [ ]:
def test_deepsort():
  # Config
  save_video = True # Creates a video from all the frames
  play_frames = False # Plays the frames while processing
  frames_path = sorted(glob.glob(dataset_path + "/test/images/*.jpg"))
  output_video_path = str(output_path / "deepsort_output.mp4")
  
  draw_boxes = DrawBoxes()

  # Validating the frames path
  if not frames_path:
      raise RuntimeError("No frames found")

  # Ensure the output directory exists
  os.makedirs(output_path, exist_ok=True)

  # Retrieve first frame dimensions
  if save_video:
    first_frame = cv2.imread(frames_path[0])
    height, width = first_frame.shape[:2]
    fps = 5

    # Define the codec and create VideoWriter
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')  # Codec for .mp4 files
    out = cv2.VideoWriter(output_video_path, fourcc, fps, (width, height))

  # Loop through images/frames
  for img_path in frames_path:

    # Read the frame
    frame = cv2.imread(img_path)
    if frame is None:
      continue

    # Process the frame
    processed_frame = frame.copy()
    
    # Run through YOLO
    yolo_results = detector.predict(frame)
    if yolo_results is None:
      continue
    
    # Convert to deepsort format and sort detections
    human_dets, object_dets = tracker.yolo_to_deepsort(yolo_results)
    
    # Run through DeepSORT
    deepsort_results = tracker.update(human_dets, frame)

    # Draw DeepSORT results
    for track in deepsort_results:
      if not track.is_confirmed():
          continue

      track_id = int(track.track_id)
      bbox = track.to_ltrb()
      class_name = track.get_det_class()
      
      # frame, bboxes, class_names, track_ids
      processed_frame = draw_boxes.draw_box(processed_frame, bbox, class_name, track_id=track_id)

    # Draw the non-human objects
    for bbox,_,class_name,class_id in object_dets:
      processed_frame = draw_boxes.draw_box(processed_frame, bbox, class_name, class_id=class_id)

    if save_video: out.write(processed_frame) # Write current frame to output
    if play_frames: cv2_imshow(processed_frame) # output current frame

    # if cv2.waitKey(1) & 0xFF == ord('q'):
      # break

  # Release resources
  if save_video: out.release()
  # cv2.destroyAllWindows()

In [ ]:
test_deepsort()